# E1 TabPFN windows on Kaggle (background run)

**One-time setup in the right-hand panel (Settings / Add-ons):**
1. **Accelerator:** `GPU T4 x2` (not P100: recent PyTorch no longer supports it).
2. **Internet:** On (needs a verified phone number on your Kaggle account).
3. **Add Input → Upload → New Dataset:** upload your latest `e1_rows.csv`.
4. **Add-ons → Secrets:** add `TABPFN_TOKEN` (your Prior Labs API key) and tick it for this notebook.

**Then:** run cell 1 and 2 once to check (`key loaded: True`, `rows: 1500` or more), and click
**Save Version → Save & Run All (Commit)**. You can close the browser. When it finishes, open the version's
**Output** tab and download `e1_rows.csv` and `versions.json`.

The run saves after each window, and only windows not already in your uploaded CSV are run.

## 0. Install

In [1]:
!pip install -q tabpfn xgboost pyarrow

## 1. Key, results file, environment

In [2]:
# 1. Key, results file and environment
import os, sys, glob, json, shutil, importlib.metadata as md
import pandas as pd

# TabPFN key: Kaggle secret, else Colab secret, else environment
key = None
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("TABPFN_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        key = userdata.get("TABPFN_TOKEN")
    except Exception:
        key = os.environ.get("TABPFN_TOKEN")
if key:
    os.environ["TABPFN_TOKEN"] = key.strip()
    os.environ["TABPFN_NO_BROWSER"] = "1"
print("key loaded:", bool(key), "| length:", len(key.strip()) if key else 0)

# Copy the uploaded results file (read-only input) to the working folder, keeping the largest copy
if not os.path.exists("e1_rows.csv"):
    found = glob.glob("/kaggle/input/**/e1_rows*.csv", recursive=True)
    if found:
        best = max(found, key=lambda f: len(pd.read_csv(f)))
        shutil.copy(best, "e1_rows.csv")
        print("copied", best)
if os.path.exists("e1_rows.csv"):
    d = pd.read_csv("e1_rows.csv")
    print("rows:", len(d)); print(d.groupby(["model", "design"]).window.unique())
else:
    print("WARNING: e1_rows.csv not found. Add it as an input dataset, or everything will run from scratch.")

versions = {"python": sys.version.split()[0]}
for p in ["numpy", "pandas", "scikit-learn", "xgboost", "tabpfn", "torch"]:
    try: versions[p] = md.version(p)
    except md.PackageNotFoundError: versions[p] = "not installed"
try:
    import torch
    versions["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
except ImportError:
    versions["gpu"] = "torch not installed"
print(versions)
json.dump(versions, open("versions.json", "w"), indent=1)

key loaded: True | length: 53
copied /kaggle/input/datasets/gianghainuk/e1-tabpfn-kaggle-ipynb/e1_rows (2).csv
rows: 1525
model          design   
Logistic       adjacent              [t=10-19, t=20-29]
               all_later    [t=10-19, t=20-29, t=30-39]
Random Forest  adjacent              [t=10-19, t=20-29]
               all_later    [t=10-19, t=20-29, t=30-39]
TabPFN         all_later                      [t=10-19]
XGBoost        adjacent              [t=10-19, t=20-29]
               all_later    [t=10-19, t=20-29, t=30-39]
Name: window, dtype: object
{'python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.3.3', 'scikit-learn': '1.6.1', 'xgboost': '3.4.1', 'tabpfn': '9.2.0', 'torch': '2.11.0+cu130', 'gpu': 'Tesla T4'}


## 2. Data and models

In [3]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

import urllib.request
from pathlib import Path
if not Path("elliptic_labelled.parquet").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
                               "main/resources/elliptic/elliptic_labelled_v2.parquet", "elliptic_labelled.parquet")
ell = pd.read_parquet("elliptic_labelled.parquet")
feat_cols = [c for c in ell.columns if c.startswith("feat_")]
X = ell[feat_cols].values
y = (ell["class"] == 1).astype(int).values
ts = ell["time_step"].values
boundaries = [1, 10, 20, 30, 40, 50]
assert len(y) == 46564 and y.sum() == 4545

def make_model(name, seed=42):
    if name == "Logistic":
        return Pipeline([("sc", StandardScaler()),
                         ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced",
                                      random_state=seed, n_jobs=-1)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                             random_state=seed, eval_metric="logloss", n_jobs=-1)
    if name == "TabPFN":
        from tabpfn import TabPFNClassifier
        # default configuration, no rebalancing ("no tuning" principle)
        return TabPFNClassifier(device="cuda", random_state=seed)
    raise ValueError(name)

def fit_proba(name, tr_idx, te_mask, seed=42):
    m = make_model(name, seed)
    ytr = y[tr_idx]
    if name == "XGBoost":   # class weight from THIS training set only (no look-ahead)
        m.set_params(scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum())
    m.fit(X[tr_idx], ytr)
    Xte = X[te_mask]
    # predict in chunks to limit GPU memory for TabPFN
    return np.concatenate([m.predict_proba(Xte[k:k + 5000])[:, 1] for k in range(0, len(Xte), 5000)])

## 3. Settings

In [4]:
# 3. Settings: only TabPFN (the other models are already in e1_rows.csv)
MODELS = ["TabPFN"]
E1_B = {"Logistic": 100, "Random Forest": 100, "XGBoost": 100, "TabPFN": 25}

## 4. E1 for TabPFN

In [5]:
# 4. E1 controlled comparison (cell 41 design); skips everything already saved
import time
from sklearn.metrics import roc_auc_score, average_precision_score
PAST_SEEDS = [42, 1, 7, 21, 99]
OUT = Path("e1_rows.csv")
done = set()
if OUT.exists():
    d = pd.read_csv(OUT)
    done = set(zip(d.model, d.design, d.window))

def strat_draw(pool_idx, n_ill, n_lic, rng):
    ill = pool_idx[y[pool_idx] == 1]; lic = pool_idx[y[pool_idx] == 0]
    return np.concatenate([rng.choice(ill, n_ill, replace=len(ill) < n_ill),
                           rng.choice(lic, n_lic, replace=len(lic) < n_lic)])

all_idx = np.arange(len(y))
for name in MODELS:
    stochastic = name != "Logistic"
    for design in ["all_later", "adjacent"]:
        if name == "TabPFN" and design == "adjacent":     # option C: skip adjacent design for TabPFN
            continue
        for i in range(len(boundaries) - 3):              # final window has no later pool
            cut, hi = boundaries[i + 1], boundaries[i + 2]
            win = f"t={cut}-{hi-1}"
            if design == "adjacent" and cut == 30:        # identical to all_later
                continue
            if (name, design, win) in done:
                continue
            t0 = time.time()
            test = (ts >= cut) & (ts < hi)
            past = ts < cut
            fut = (~test) & (ts >= cut) if design == "all_later" else (ts >= hi) & (ts < hi + 10)
            past_idx = all_idx[past]
            seeds = PAST_SEEDS if stochastic else [42]
            pp = [fit_proba(name, past_idx, test, s) for s in seeds]
            auc_p = np.mean([roc_auc_score(y[test], p) for p in pp])
            ap_p = np.mean([average_precision_score(y[test], p) for p in pp])
            n_ill, n_lic = int(y[past].sum()), int((y[past] == 0).sum())
            pool, later = all_idx[past | fut], all_idx[fut]
            rows = []
            for b in range(E1_B[name]):
                rng = np.random.default_rng(1000 + b)     # same draws as cell 41
                draw = strat_draw(pool, n_ill, n_lic, rng)
                p = fit_proba(name, draw, test, seed=b if stochastic else 42)
                rows.append({"model": name, "design": design, "window": win, "b": b,
                             "auc_past": auc_p, "ap_past": ap_p,
                             "auc_fi": roc_auc_score(y[test], p),
                             "ap_fi": average_precision_score(y[test], p),
                             "share_later": np.isin(draw, later).mean(),
                             "past_n": int(past.sum()), "later_pool_n": int(fut.sum())})
            pd.DataFrame(rows).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
            r = pd.DataFrame(rows)
            print(f"{name:13s} {design:9s} {win}: dAUC={(r.auc_fi - auc_p).mean():+.4f} "
                  f"dAP={(r.ap_fi - ap_p).mean():+.4f} ({time.time() - t0:.0f}s)", flush=True)
print("E1 done", flush=True)
d = pd.read_csv(OUT); print("final rows:", len(d)); print(d.groupby(["model", "design"]).window.unique())

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

TabPFN        all_later t=20-29: dAUC=+0.0781 dAP=+0.1459 (5204s)
TabPFN        all_later t=30-39: dAUC=+0.0030 dAP=+0.0113 (8299s)
E1 done
final rows: 1575
model          design   
Logistic       adjacent              [t=10-19, t=20-29]
               all_later    [t=10-19, t=20-29, t=30-39]
Random Forest  adjacent              [t=10-19, t=20-29]
               all_later    [t=10-19, t=20-29, t=30-39]
TabPFN         all_later    [t=10-19, t=20-29, t=30-39]
XGBoost        adjacent              [t=10-19, t=20-29]
               all_later    [t=10-19, t=20-29, t=30-39]
Name: window, dtype: object
